In [1]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [2]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [3]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_fraud.json")

##### Rules

In [4]:
Rule1 = Rule('''
MATCH (c:Client)
WHERE NOT c:Mule
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 2000
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns),
    names = c.name,
    namess = c.name,
    namesss = c.name,
})
''', env=env, type_strict=False)

Rule2 = Rule('''
MATCH (c:Client:Mule)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 2000
GENERATE
(p = (c.id):Scammer {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
''', env=env, type_strict=True)

Rule3 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashIn)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_CASHIN]->((t.id):CashIn {
        original_amount = t.amount,
        formatted_amount = round(t.amount * 100) / 100.0,
        is_large_amount = t.amount > 150000
    })
''', env=env, type_strict=True)

Rule4 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashOut)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_CASHOUT]->(tx = (t.id):CashOut {
        original_amount = t.amount,
        formatted_amount = round(t.amount * 100) / 100.0,
        is_large_amount = t.amount > 150000
    })
''', env=env, type_strict=False)

Rule5 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Payment)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_PAYMENT]->(tx = (t.id):Payment)
''', env=env, type_strict=True)

Rule6 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Transfer)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_TRANSFER]->(tx = (t.id):Transfer)
''', env=env, type_strict=True)

Rule7 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Debit)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_DEBITS]->(tx = (t.id):Debit)
''', env=env, type_strict=True)

In [ ]:
from dtgraph.type_checking.check_types import check_types

check_types([Rule1], env)

##### Applying Rules

In [11]:
my_transform = Transformation([Rule1, Rule2])
my_transform.apply_on(graph)

Index: Added 1 index, completed after 6 ms.
self._dict:  {'lhs': 'MATCH (c:Client)\nOPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)\nOPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)\nOPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)\nWHERE NOT c:Mule AND c.id IS NOT NULL AND c.name IS NOT NULL\nWITH c,\n    collect(DISTINCT e.email) AS emails,\n    collect(DISTINCT p.phoneNumber) AS phones,\n    collect(DISTINCT s.ssn) AS ssns\nLIMIT 2000', 'constructors': [{'alias': 'p', 'ids': ['c.id'], 'labels': ['Person'], 'properties': [{'key': 'id', 'value': 'c.id'}, {'key': 'name', 'value': 'c.name'}, {'key': 'name_camel_case', 'value': 'apoc.text.upperCamelCase(c.name)'}, {'key': 'email', 'value': 'head(emails)'}, {'key': 'phone', 'value': 'head(phones)'}, {'key': 'ssn', 'value': 'head(ssns)'}, {'key': 'names', 'value': 'c.name'}, {'key': 'namess', 'value': 'c.name'}, {'key': 'namesss', 'value': 'c.name'}]}]} 

Before LHS Validation:
 MATCH (c:Client)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:H

Rule: Added 4000 labels, created 2000 nodes, set 20000 properties, created 0 relationships, completed after 840 ms.
self._dict:  {'lhs': 'MATCH (c:Client:Mule)\nOPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)\nOPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)\nOPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)\nWHERE c.id IS NOT NULL AND c.name IS NOT NULL\nWITH c,\n    collect(DISTINCT e.email) AS emails,\n    collect(DISTINCT p.phoneNumber) AS phones,\n    collect(DISTINCT s.ssn) AS ssns\nLIMIT 2000', 'constructors': [{'alias': 'p', 'ids': ['c.id'], 'labels': ['Scammer'], 'properties': [{'key': 'id', 'value': 'c.id'}, {'key': 'name', 'value': 'c.name'}, {'key': 'name_camel_case', 'value': 'apoc.text.upperCamelCase(c.name)'}, {'key': 'email', 'value': 'head(emails)'}, {'key': 'phone', 'value': 'head(phones)'}, {'key': 'ssn', 'value': 'head(ssns)\n'}]}]} 

Before LHS Validation:
 MATCH (c:Client:Mule)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HA

1421

##### Abort Transformation

In [12]:
my_transform.abort()

Index: Removed 1 index, completed after 10 ms.
Abort: Deleted 2377 nodes, deleted 0 relationships, completed after 45 ms.
